In [64]:
!pip3 install crewai crewai-tools langchain-community langchain-huggingface sentence-transformers litellm --break-system-packages

In [65]:
from crewai import Agent, Task, Crew, Process
from crewai import LLM
from crewai_tools import PDFSearchTool, SerperDevTool
import litellm
litellm.ssl_verify = False

### Agentic AI with CrewAI

Instead of hard-coding rigid chatbot logic, CrewAI lets you delegate work to a coordinated team of autonomous agents.  
Each agent is defined by a **role**, **goal**, **backstory**, and **tools**, while **tasks** specify what they must do.  
The **Crew** orchestrates agents and task execution order to produce the final outcome.

##### The Role of Tools: Agent-Centric vs Task-Centric

**1. Agent-Centric Approach (Generalist):**  
Give an agent a broad toolbox and let it choose tools dynamically. This is flexible, but can be less efficient or more error-prone.

![Diagram 1](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/NPYpo2eWUERulpzrm9xYtw/Adobe%20Express%20-%20file.jpg)

**2. Task-Centric Approach (Specialist):**  
Attach tools to specific tasks instead of the agent. The agent only gets task-relevant tools during execution, making behavior more focused and predictable.

![Diagram 2.jpg](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/olP9UG8s4B7QnOJhCRH4aA/Diagram%202.jpg)

### Writing the Code

##### Setup the LLM

In [66]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [67]:
from crewai import LLM

llm = LLM(
    model="huggingface/Qwen/Qwen3.5-397B-A17B:novita",
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    base_url="https://router.huggingface.co/v1",
    max_retries=7,           # total retry attempts
    timeout=300,             # seconds per request
    retry_min_wait=4,        # initial backoff seconds
    retry_max_wait=35
)

##### Defining Tools for Information Gathering

The chatbot uses two sources: a Daily Dish FAQ PDF and the web.  
1. **`PDFSearchTool`** indexes and semantically searches the FAQ PDF for relevant sections.  
2. **`SerperDevTool`** runs web searches via SerperAPI.  

To use `SerperDevTool`, get an API key from `serper.dev` (Dashboard → API Keys) and add it to your workflow config.  

Docs: https://serper.dev/

In [68]:
web_search_tool = SerperDevTool()

In [69]:
import os
import tempfile

# Force sqlite temp files to a writable location
os.environ["SQLITE_TMPDIR"] = "/tmp"

# Fresh writable Chroma path for this session
auto_chroma_dir = tempfile.mkdtemp(prefix="chroma_rw_")
print("Using Chroma dir:", auto_chroma_dir)

Using Chroma dir: /var/folders/td/rtvtvvjn3x1gfqswnc7bb6hm0000gn/T/chroma_rw_x1jekyvg


In [70]:
import warnings
from datetime import datetime
from chromadb.config import Settings
from crewai_tools import PDFSearchTool

warnings.filterwarnings("ignore")  # Keeps Jupyter Notebook clean

pdf_search_tool = PDFSearchTool(
    pdf="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf",
    collection_name=f"rag_crewai_pdf",
    config={
        "embedding_model": {
            "provider": "sentence-transformer",
            "config": {
                "model_name": "all-MiniLM-L6-v2",
                "device": "cpu",
                "normalize_embeddings": False,
            },
        },
        "vectordb": {
            "provider": "chromadb",
            "config": {
                "settings": Settings(
                    persist_directory=auto_chroma_dir,
                    is_persistent=True,
                    allow_reset=True,
                )
            },
        },
    },
)

### Approach 1: Standard Method (Agent-Centric Tools)

We first build the chatbot with the conventional setup: the agent gets both tools and decides when to use each (PDF search or direct LLM response).

##### Step 1.1: Create the Agent
Define an `Inquiry Specialist Agent` for question answering, with `tools=[pdf_search_tool, web_search_tool]`.

In [71]:
agent_centric_agent = Agent(
    role="The Daily Dish Inquiry Specialist",
    goal="""Accurately answer customer questions about The Daily Dish restaurant. 
    You must decide whether to use the restaurant's FAQ PDF or a web search to find the best answer.""",
    backstory="""You are an AI assistant for 'The Daily Dish'.
    You have access to two tools: one for searching the restaurant's FAQ document and another for searching the web.
    Your job is to analyze the user's question and choose the most appropriate tool to find the information needed to provide a helpful response.""",
    tools=[pdf_search_tool, web_search_tool],
    verbose=True,
    allow_delegation=False,
    llm=llm
)

##### **Step 1.2: Define the Task**

We create a single, broad task that instructs the agent to handle the customer's query.

In [72]:
agent_centric_task = Task(
    description="Answer the following customer query: '{customer_query}'. "
                "Analyze the question and use the tools at your disposal (PDF search or web search) to find the most relevant information. "
                "Synthesize the findings into a clear and friendly response.",
    expected_output="A comprehensive and well-formatted answer to the customer's query.",
    agent=agent_centric_agent
)

##### **Step 1.3: Assemble the Crew**

Finally, we create the Crew. It's a simple setup with our one agent and one task.

In [73]:
# Download the FAQ document for the tool to use
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf

--2026-03-15 14:15:29--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.45.118.108
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.45.118.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53993 (53K) [application/pdf]
Saving to: ‘The-Daily-Dish-FAQ.pdf.1’

The-Daily-Dish-FAQ. 100%[===================>]  52.73K   127KB/s    in 0.4s    

2026-03-15 14:15:31 (127 KB/s) - ‘The-Daily-Dish-FAQ.pdf.1’ saved [53993/53993]



In [74]:
agent_centric_crew = Crew(
    agents=[agent_centric_agent],
    tasks=[agent_centric_task],
    process=Process.sequential,
    verbose=False
)

In [76]:
print("\nWelcome to The Daily Dish Chatbot!")
print("What would you like to know? (Type 'exit' to quit)")

while True: 
    user_input = input("\nYour question: ").lower()
    if user_input == 'exit':
        print("Thank you for chatting. Have a great day!")
        break
    
    if not user_input:
        print("Please type a question.")
        continue

    try:
        # Here we use our more advanced, task-centric crew
        result_agent_centric = agent_centric_crew.kickoff(inputs={'customer_query': user_input})
        print("\n--- The Daily Dish Assistant ---")
        print(result_agent_centric)
        print("--------------------------------")
    except Exception as e:
        print(f"An error occurred: {e}")


Welcome to The Daily Dish Chatbot!
What would you like to know? (Type 'exit' to quit)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Daily Dish Inquiry Specialist                                                                       │
│                                                                                                                 │
│  Task: Answer the following customer query: 'what are your best dishes?'. Analyze the question and use the      │
│  tools at your disposal (PDF search or web search) to find the most relevant information. Synthesize the        │
│  findings into a clear and friendly response.                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Relevant Content:                                                                                              │
│                                                                                                                 │
│  when making your reservation.                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Menu & Dietary Needs                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  9.  Q: What type of cuisine do you offer?                                                                      │
│                                                                                                                 │
│      A: The Daily Dish features a contemporary American menu with a focus on fresh,                             │
│                                                                                                                 │
│  seasonal ingredients and global influences.                                                                    │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  10. Q: What is your most popular dish?                                                                         │
│                                                                                                                 │
│      A: Our "Chef's Signature Salmon with Lemon-Dill Risotto" and the "Spicy Chorizo &                          │
│                                                                                                                 │
│  Manchego Flatbread" are consistent guest favorites!                                                            │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  11. Q: Do you have vegetarian or vegan options?                                                                │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                       

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Daily Dish Inquiry Specialist                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Thank you for your interest in The Daily Dish! We're delighted to share our most popular dishes with you.      │
│                                                                                                                 │
│  **Our Guest Favorites:**                                                                                       │
│                                                                                                                 │
│  According to our FAQ, our two most popular and consistently loved dishes are:                                  │
│                                                                                                                 │
│  1. **Chef's Signature Salmon with Lemon-Dill Risotto** - A beautifully prepared salmon dish paired with        │
│  creamy, aromatic lemon-dill risotto that keeps our guests coming back for more.                                │
│                                                                                                                 │
│  2. **Spicy Chorizo & Manchego Flatbread** - A flavorful flatbread featuring spicy chorizo and rich Manchego    │
│  cheese that has become a customer favorite.                                                                    │
│                                                                                                                 │
│  **Additional Highlights:**                                                                                     │
│                                                                                                                 │
│  - **Daily Dish Specials**: Our chef creates unique daily specials based on the freshest market ingredients     │
│  available. Be sure to ask your server or check our specials board for these rotating offerings!                │
│                                                                                                                 │
│  - **Dietary Options**: We have a dedicated section on our menu for vegetarian and vegan dishes, and many       │
│  other items can be modified to suit your needs. We also offer a variety of gluten-free options including       │
│  pasta, bread, and desserts.                                                                                    │
│                                                                                                                 │
│  - **Happy Hour**: Don't miss our "Dish Delights Happy Hour" running Monday to Friday from 4:00 PM to 6:00 PM,  │
│  featuring discounted drinks and appetizers.                                                                    │
│                                                                                                                 │
│  Our cuisine features contemporary American dishes with a focus on fresh, seasonal ingredients and global       │
│  influences. We'd love to welcome you to experience these delicious dishes firsthand! You can view our          │
│  complete menu with prices on our website, or feel free to call us at (555) 123-4567 for more information.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- The Daily Dish Assistant ---
Thank you for your interest in The Daily Dish! We're delighted to share our most popular dishes with you.

**Our Guest Favorites:**

According to our FAQ, our two most popular and consistently loved dishes are:

1. **Chef's Signature Salmon with Lemon-Dill Risotto** - A beautifully prepared salmon dish paired with creamy, aromatic lemon-dill risotto that keeps our guests coming back for more.

2. **Spicy Chorizo & Manchego Flatbread** - A flavorful flatbread featuring spicy chorizo and rich Manchego cheese that has become a customer favorite.

**Additional Highlights:**

- **Daily Dish Specials**: Our chef creates unique daily specials based on the freshest market ingredients available. Be sure to ask your server or check our specials board for these rotating offerings!

- **Dietary Options**: We have a dedicated section on our menu for vegetarian and vegan dishes, and many other items can be modified to suit your needs. We also offer a variety of glut

### Approach 2: A More Focused Method (Task-Centric Tools)

Now, we'll refactor our solution to use a task-centric approach. We will create a multi-step process where each step (Task) has its own dedicated tool. This makes the agent's job simpler and the overall workflow more reliable.

Our new workflow will have two tasks:
1.  **Search the FAQ:** This task will *only* use the `PDFSearchTool`.
2.  **Draft the Response:** This task will use the information from the first two tasks to write the final answer. It needs no tools.

##### **Step 2.1: Create the Agent**

This time, we create a `Customer Service Specialist` agent. Notice the critical difference: the `tools` list is **empty**. We are not giving the agent its toolbox upfront. However, even if we gave the agent many many tools, they would simply be completely overriden by the tools in the task.


In [ ]:
task_centric_agent = Agent(
    role="Customer Service Specialist",
    goal="Provide exceptional customer service by following a multi-step process to answer customer questions accurately.",
    backstory="""You are an AI assistant for 'The Daily Dish'.
    You are an expert at following instructions. You will be given a sequence of tasks to complete.
    For each task, you will be provided with the specific tool needed to accomplish it.
    Your job is to execute each task diligently and pass the results to the next step.""",
    tools=[], # The agent is not given any tools directly
    verbose=True,
    allow_delegation=False,
    llm=llm
)

##### **Step 2.2: Define the Tasks with Specific Tools**

Here is the core of the new approach. We define two distinct tasks. For the first two, we use the `tools` parameter within the `Task` definition itself to assign a specific tool to that step.

- **`faq_search_task`**: Is exclusively paired with `pdf_search_tool`.
- **`response_drafting_task`**: Needs the output (context) from the previous tasks but requires no tools of its own.


In [ ]:
faq_search_task = Task(
    description="Search the restaurant's FAQ PDF for information related to the customer's query: '{customer_query}'.",
    expected_output="A snippet of the most relevant information from the PDF, or a statement that the information was not found.",
    tools=[pdf_search_tool], # Tool assigned directly to the task
    agent=task_centric_agent
)

response_drafting_task = Task(
    description="Using the information gathered from the FAQ search, draft a friendly and comprehensive response to the customer's query: '{customer_query}'.",
    expected_output="The final, customer-facing response.",
    agent=task_centric_agent,
    context=[faq_search_task]
)

##### **Step 2.3: Assemble the New Crew**

We assemble our new crew, providing the single agent and the list of three tasks. The `Process.sequential` setting ensures the tasks run in the order we've listed them.


In [ ]:
task_centric_crew = Crew(
    agents=[task_centric_agent],
    tasks=[faq_search_task, response_drafting_task],
    process=Process.sequential,
    verbose=True
)

In [ ]:
print("\nWelcome to The Daily Dish Chatbot!")
print("What would you like to know? (Type 'exit' to quit)")

while True: 
    user_input = input("\nYour question: ").lower()
    if user_input == 'exit':
        print("Thank you for chatting. Have a great day!")
        break
    
    if not user_input:
        print("Please type a question.")
        continue

    try:
        # Here we use our more advanced, task-centric crew
        result_task_centric = task_centric_crew.kickoff(inputs={'customer_query': user_input})
        print("\n--- The Daily Dish Assistant ---")
        print(result_task_centric)
        print("--------------------------------")
    except Exception as e:
        print(f"An error occurred: {e}")


Welcome to The Daily Dish Chatbot!
What would you like to know? (Type 'exit' to quit)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ea5ec57c-0152-4cb6-b7ee-ea0a7e64adf1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the restaurant's FAQ PDF for information related to the customer's query: 'what is the            │
│  location?'.                                                                                                    │
│  ID: c349db59-df44-47d1-814a-eab57628c665                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Service Specialist                                                                             │
│                                                                                                                 │
│  Task: Search the restaurant's FAQ PDF for information related to the customer's query: 'what is the            │
│  location?'.                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_a_pdfs_content                                                                                    │
│  Args: {"query": "what is the location"}                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the information gathered from the FAQ search, draft a friendly and comprehensive response to the   │
│  customer's query: 'what is the location?'.                                                                     │
│  Agent: Customer Service Specialist                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ea5ec57c-0152-4cb6-b7ee-ea0a7e64adf1                                                                       │
│  Final Output: Thank you for your interest in The Daily Dish! We'd be delighted to welcome you to our           │
│  restaurant.                                                                                                    │
│                                                                                                                 │
│  Our location is:                                                                                               │
│                                                                                                                 │
│  **123 Culinary Avenue, Foodie Town, FT 54321**                                                                 │
│                                                                                                                 │
│  We're conveniently situated in the heart of Foodie Town with complimentary valet parking available, as well    │
│  as street parking nearby.                                                                                      │
│                                                                                                                 │
│  If you need any additional information about our hours of operation, menu, or would like to make a             │
│  reservation, please don't hesitate to reach out to us at (555) 123-4567 or visit our website.                  │
│                                                                                                                 │
│  We look forward to serving you soon!                                                                           │
│                                                                                                                 │
│  Warm regards,                                                                                                  │
│  The Daily Dish Team                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- The Daily Dish Assistant ---
Thank you for your interest in The Daily Dish! We'd be delighted to welcome you to our restaurant.

Our location is:

**123 Culinary Avenue, Foodie Town, FT 54321**

We're conveniently situated in the heart of Foodie Town with complimentary valet parking available, as well as street parking nearby.

If you need any additional information about our hours of operation, menu, or would like to make a reservation, please don't hesitate to reach out to us at (555) 123-4567 or visit our website.

We look forward to serving you soon!

Warm regards,
The Daily Dish Team
--------------------------------


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thank you for chatting. Have a great day!


### Conclusion

In this lab, you built a customer service chatbot and learned a key CrewAI concept: how tool assignment changes workflow behavior.

You compared two patterns:
1. **Agent‑Centric:** quick to set up and flexible, but tool selection depends on LLM reasoning and can be inconsistent.  
2. **Task‑Centric:** structured and reliable, with tools explicitly attached to tasks for deterministic execution.

| Aspect             | Agent‑Centric Output                                                   | Task‑Centric Output                                                       |
|--------------------|-------------------------------------------------------------------------|----------------------------------------------------------------------------|
| **Predictability** | Varies run‑to‑run based on the LLM’s tool choice and phrasing.          | Consistent across runs thanks to fixed task‑to‑tool mapping.              |
| **Debuggability**  | Harder to trace—calls and errors are buried in the LLM’s reasoning.     | Easy to debug—each task’s inputs, outputs, and errors are explicitly logged. |
| **Reusability**    | You get a free‑form text blob that you must parse yourself.             | You get structured intermediate results (for example, JSON or code blocks) ready for reuse. |
| **Structure**¹     | Search and formatting are blended in one step.                          | A separate formatting task produces a clean, structured final message.    |

This “Structure” difference exists only because we added a dedicated formatting task in the Task‑Centric workflow.

**Key advantages of Task‑Centric tools:**
- **Focus & Efficiency:** agents use only the required tool per task.  
- **Clarity & Maintainability:** task-to-tool mapping is explicit and easy to follow.  
- **Control & Security:** sensitive tools can be scoped to specific tasks only.

Agent‑centric still has valid use cases, but task‑centric design is typically stronger for production-grade multi-agent systems.

##### Extending CrewAI with Custom Functions

In CrewAI, **tools** are callable functions agents use for actions like search, document reading, or calculations.  
Besides built-in tools, you can create custom ones by wrapping Python functions with `@tool` from `crewai.tools`.

To define a custom tool:
- decorate with `@tool("Tool Name")`  
- include a clear docstring (used by the LLM to understand usage)  
- implement the function logic

This is similar in concept to LangChain’s `@tool` pattern.

In [77]:
from crewai.tools import tool
import re

@tool("Add Two Numbers Tool")
def add_numbers(data: str) -> int:
    """
    Extracts and adds integers from the input string.
    Example input: 'add 1 and 2' or '[1,2,3,4]'
    Output: sum of the numbers
    """
    # Find all integers in the string
    numbers = list(map(int, re.findall(r'-?\d+', data)))
    return sum(numbers)

In [78]:
from functools import reduce

@tool("Multiply Numbers Tool")
def multiply_numbers(data: str) -> int:
    """
    Extracts and multiplies integers from the input string.
    Example input: 'multiply 2 and 3' or '[2,3,4]'
    Output: the product of all numbers found
    """
    numbers = list(map(int, re.findall(r'-?\d+', data)))
    return reduce(lambda x, y: x * y, numbers, 1)

In [79]:
calculator_agent = Agent(
    role="Calculator",
    goal="Extracts, adds, or multiplies numbers when asked, using the Add Two Numbers and Multiply Numbers tools.",
    backstory="An expert at parsing numeric instructions and computing sums or products.",
    tools=[add_numbers, multiply_numbers],
    llm=llm,
    allow_delegation=False
)

In [80]:
calculation_task = Task(
    description="Extract numbers from '{numbers}' and either add or multiply them, depending on the natural-language instruction.",
    expected_output="An integer result (sum or product) based on the user’s request.",
    agent=calculator_agent
)

In [81]:
crew = Crew(
    agents=[calculator_agent],
    tasks=[calculation_task],
    # verbose=True #Uncomment this to see the steps taken to get the final answer
)

In [82]:
# Inputs for addition…
result = crew.kickoff(inputs={'numbers': 'please add 4, 5, and 6'})
print("Sum result:", result)

Sum result: 15


In [83]:
# Inputs for multiplication…
result = crew.kickoff(inputs={'numbers': 'multiply 7 and 8 also 9 dont forget 10'})
print("Product result:", result)

Product result: 5040
